## Downloading the Data

In [2]:
!pip install pdfminer.six
from pdfminer.high_level import extract_pages, extract_text
from pdfminer.layout import LTTextContainer, LTChar
from google.colab import drive
from google.colab import userdata
import re


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 145.8 MB/s eta 0:00:00


In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path_to_folder = f'{userdata.get('PROJECTS_FOLDER_PATH')}/PostgreSQL AI Documentation Assistant'
documentation_pdf_file_path = f'{path_to_folder}/postgresql-18-documentation.pdf'

# Read from pdfminer.six
documentation_pdf = list(extract_pages(documentation_pdf_file_path))
print(f"Extracted {len(documentation_pdf)} pages.")

Extracted 3307 pages.


In [5]:
extracted_text = extract_text(documentation_pdf_file_path)

cleaned_text = re.sub(r'\n+', '\n', extracted_text) # Replace multiple newlines with a single newline
cleaned_text = re.sub(r' +', ' ', cleaned_text) # Replace multiple spaces with a single space
cleaned_text = cleaned_text.strip() # Remove leading/trailing whitespace

print(f"Extracted {len(cleaned_text)} characters after cleaning.")
# Display the first 1000 characters of the cleaned text to verify
print(cleaned_text[:1000])

Extracted 7410575 characters after cleaning.
PostgreSQL 18.4 Documentation
The PostgreSQL Global Development Group
PostgreSQL 18.4 Documentation
The PostgreSQL Global Development Group
Copyright © 1996–2026 The PostgreSQL Global Development Group
Legal Notice
PostgreSQL Database Management System (also known as Postgres, formerly known as Postgres95)
Portions Copyright © 1996-2026, PostgreSQL Global Development Group
Portions Copyright © 1994, The Regents of the University of California
Permission to use, copy, modify, and distribute this software and its documentation for any purpose, without fee, and without a written agreement
is hereby granted, provided that the above copyright notice and this paragraph and the following two paragraphs appear in all copies.
IN NO EVENT SHALL THE UNIVERSITY OF CALIFORNIA BE LIABLE TO ANY PARTY FOR DIRECT, INDIRECT, SPECIAL, INCI-
DENTAL, OR CONSEQUENTIAL DAMAGES, INCLUDING LOST PROFITS, ARISING OUT OF THE USE OF THIS SOFTWARE AND ITS
DOCUMENTATION,

Now that `pdfminer.six` is installed, we can use it to extract text while also trying to identify structural elements like headers. We'll iterate through each page and extract text elements along with their font sizes. Headers often have larger font sizes than regular body text, which can serve as a basic heuristic for differentiation. This initial step will gather the text with its properties, and in a later step, we can apply specific rules to categorize them.

Let's define some heuristics to classify text elements as potential queries, headers, or body text based on `font_name`, `font_size`, and `bbox` properties. We'll start with a simple classification and can refine it as needed.

## Extracting Metadata for Lines of Text

In [6]:
def global_round(value, precision=3):
  return round(value, precision)

In [7]:
from dataclasses import dataclass

@dataclass
class Line:
    text: str
    page: int
    font_name: str
    font_size: float
    bbox: tuple
    classification: str | None = None

In [8]:
def extract_text_with_metadata(pdf_path):
    all_elements = []

    for page_num, page_layout in enumerate(extract_pages(pdf_path), start=1):

        for element in page_layout:

            if isinstance(element, LTTextContainer):

                for text_line in element:

                    line_text = text_line.get_text().strip()

                    if not line_text:
                        continue

                    font_size = None
                    font_name = None

                    if hasattr(text_line, "_objs") and text_line._objs:
                        first_char = text_line._objs[0]

                        if isinstance(first_char, LTChar):
                            font_size = first_char.size
                            font_name = first_char.fontname

                    all_elements.append({
                        "text": line_text,
                        "page": page_num,
                        "font_size": global_round(font_size), # Account for floating-point precision
                        "font_name": font_name,
                        "bbox": tuple(global_round(e) for e in text_line.bbox),
                        "classification": None
                    })

    return all_elements

pdf_file_path = f'{path_to_folder}/postgresql-18-documentation.pdf'
text_elements = extract_text_with_metadata(pdf_file_path)


print(f"Extracted {len(text_elements)} text elements with metadata.")

# Display the first few extracted elements to see their structure
for i, element in enumerate(text_elements[:10]):
    print(f"Element {i+1}: {element}")

Extracted 136970 text elements with metadata.
Element 1: {'text': 'PostgreSQL 18.4 Documentation', 'page': 1, 'font_size': 24.883, 'font_name': 'Helvetica-Bold', 'bbox': (112.432, 693.562, 499.587, 718.445), 'classification': None}
Element 2: {'text': 'The PostgreSQL Global Development Group', 'page': 1, 'font_size': 17.28, 'font_name': 'Helvetica-Bold', 'bbox': (124.051, 527.781, 487.968, 545.061), 'classification': None}
Element 3: {'text': 'PostgreSQL 18.4 Documentation', 'page': 2, 'font_size': 14.4, 'font_name': 'Helvetica-Bold', 'bbox': (72.0, 704.7, 296.05, 719.1), 'classification': None}
Element 4: {'text': 'The PostgreSQL Global Development Group', 'page': 2, 'font_size': 10.0, 'font_name': 'Times-Roman', 'bbox': (72.0, 692.22, 253.09, 702.22), 'classification': None}
Element 5: {'text': 'Copyright © 1996–2026 The PostgreSQL Global Development Group', 'page': 2, 'font_size': 10.0, 'font_name': 'Times-Roman', 'bbox': (72.0, 680.22, 353.75, 690.22), 'classification': None}
Eleme

In [9]:
lines = [
    Line(
        text=e["text"],
        page=e["page"],
        font_name=e["font_name"],
        font_size=e["font_size"],
        bbox=e["bbox"],
        classification=e["classification"]
    )
    for e in text_elements
]

# The numerical components are already rounded
print(f"Extracted {len(lines)} lines.")

# Display the first few lines to verify
for i, line in enumerate(lines[:10]):
    print(f"Line {i+1}: {line}")

Extracted 136970 lines.
Line 1: Line(text='PostgreSQL 18.4 Documentation', page=1, font_name='Helvetica-Bold', font_size=24.883, bbox=(112.432, 693.562, 499.587, 718.445), classification=None)
Line 2: Line(text='The PostgreSQL Global Development Group', page=1, font_name='Helvetica-Bold', font_size=17.28, bbox=(124.051, 527.781, 487.968, 545.061), classification=None)
Line 3: Line(text='PostgreSQL 18.4 Documentation', page=2, font_name='Helvetica-Bold', font_size=14.4, bbox=(72.0, 704.7, 296.05, 719.1), classification=None)
Line 4: Line(text='The PostgreSQL Global Development Group', page=2, font_name='Times-Roman', font_size=10.0, bbox=(72.0, 692.22, 253.09, 702.22), classification=None)
Line 5: Line(text='Copyright © 1996–2026 The PostgreSQL Global Development Group', page=2, font_name='Times-Roman', font_size=10.0, bbox=(72.0, 680.22, 353.75, 690.22), classification=None)
Line 6: Line(text='Legal Notice', page=2, font_name='Times-Bold', font_size=12.0, bbox=(72.0, 654.09, 136.32, 66

## Classifying Text

In [10]:
from statistics import mode

body_font_size = mode(
    l.font_size
    for l in lines
    if l.font_size
)

print(f'Body font size: {body_font_size}')

body_font = mode(
    l.font_name
    for l in lines
    if l.font_name
)

print(f'Body font: {body_font}')

Body font size: 10.0
Body font: Times-Roman


In [11]:
import re # Ensure re is imported for regex operations

# Helper for checking separator lines, including those with pipe characters
def is_separator_line(line_text):
    stripped_text = line_text.strip()
    if not stripped_text:
        return False
    # Check if all non-whitespace characters are one of '-', '=', '|'
    # Also ensure it contains at least one '-' or '='
    return all(c in '-=|' for c in stripped_text) and any(c in '-=' for c in stripped_text)


def classify_text_element(text_element, all_elements=None, current_index=None):
    # element is expected to be a Line object, as passed from the main loop and recursive calls
    text = text_element.text
    font_size = text_element.font_size
    font_name = text_element.font_name
    bbox = text_element.bbox

    # Heuristic 0: Specific handling for 'Note' and 'Tip' lines
    normalized_text = text.strip().lower()
    if normalized_text == 'note' or normalized_text == 'tip':
        return 'note_marker'

    # Heuristic 2: Identify Headers (larger font sizes)
    if font_size and font_size >= 17.0 and 'Bold' in font_name:
        return 'header'

    # Heuristic 3: Identify Sub-headers/Section titles
    if font_size and font_size >= 12.0 and font_size < 17.0 and 'Bold' in font_name:
        return 'subheader'

    # Heuristic 1: Identify potential queries (this needs to be more precise for Courier font)
    if font_name == 'Courier' and font_size == body_font_size and bbox and bbox[0] > 110:
        text_lower = text.lower()
        text_strip = text.strip()

        # Check if current line is a separator itself (e.g., '---' or '-------|-------')
        if is_separator_line(text_strip):
            return 'output' # A separator is output

        # New Heuristic: Identify output table headers (contains '|' and is followed by a separator line)
        if all_elements is not None and current_index is not None and current_index + 1 < len(all_elements):
            next_element = all_elements[current_index + 1]
            # next_element is guaranteed to be a Line object
            next_line_text_strip = next_element.text.strip()

            if '|' in text_strip and is_separator_line(next_line_text_strip):
                return 'output' # This is an output header.
        # Now check if previous line is an output
        if all_elements is not None and current_index is not None and current_index - 1 > 0 and current_index - 1 < len(all_elements):
            prev_element = all_elements[current_index - 1]
            # prev_element is guaranteed to be a Line object
            # The recursive call classify_text_element will also expect Line objects, which it gets.
            if classify_text_element(prev_element, all_elements, current_index - 1) == 'output':
                return 'output'


        # Define patterns that are characteristic of syntax definitions/examples,
        # which should *not* be classified as executable queries.
        is_syntax_definition = False

        # Pattern 1: Contains [ and | together (strong indicator of syntax definition)
        if '[' in text_strip and '|' in text_strip:
            is_syntax_definition = True
        # Pattern 2: Looks like a single opening/closing bracket or brace
        elif text_strip in ['[', ']', '{', '}']:
            is_syntax_definition = True

        if is_syntax_definition:
            return 'body_text'

        # More precise strong SQL keywords/syntax that usually start an executable statement.
        definitive_sql_commands = [
            'select', 'insert into', 'update', 'delete from', 'create', 'alter', 'drop',
            'with', 'grant', 'revoke', 'explain', 'set transaction', 'begin', 'commit', 'rollback',
            'copy', 'fetch', 'move', 'prepare', 'execute', 'deallocate', 'declare', 'close',
            'listen', 'notify', 'unlisten', 'do', 'call'
        ]

        # Check if the line starts with a definitive SQL command
        is_definitive_command = False
        for cmd in definitive_sql_commands:
            if re.match(r'^\s*' + re.escape(cmd), text_lower):
                is_definitive_command = True
                break

        if is_definitive_command:
            # If it starts with a definitive command, it's likely a query.
            # However, if it also contains '|', '$', or ':' it's more likely a syntax definition or descriptive text.
            if '|' not in text_lower and '$' not in text_lower and ':' not in text_lower:
                return 'query'
            else: # Starts with a command, but has '|', '$', or ':' -> likely syntax or description
                return 'body_text'

        # Special handling for 'where' as it can be ambiguous
        if text_lower.startswith('where'):
            sql_condition_indicators = ['=', '>', '<', '!=', '>=', '<=', 'like', 'ilike', 'in', 'exists', 'and', 'or', '(', ')', 'is null', 'is not null']
            descriptive_phrases = ['can be one of', 'is used for', 'refers to', 'means that', 'describes how']

            has_sql_indicator = any(op in text_lower for op in sql_condition_indicators)
            has_descriptive_phrase = any(phrase in text_lower for phrase in descriptive_phrases)

            # Only classify as query if it has a SQL indicator AND doesn't look like a descriptive phrase
            if has_sql_indicator and not has_descriptive_phrase and '$' not in text_lower and ':' not in text_lower:
                return 'query'

    # Default: Body Text (standard font size and type)
    return 'body_text'

# Apply the classification to all text elements (re-run this part to apply changes)
# This part of the code needs to be re-executed after modifying classify_text_element
# so that the changes take effect for 'classified_elements' and subsequent steps.
classified_elements = []
for i, element_obj in enumerate(lines): # Iterate over lines (Line objects) and get index
    classification = classify_text_element(element_obj, all_elements=lines, current_index=i)
    classified_elements.append(
        {
            'text': element_obj.text,
            'page': element_obj.page,
            'font_size': element_obj.font_size,
            'font_name': element_obj.font_name,
            'bbox': element_obj.bbox,
            'classification': classification
        }
    )

print("First 20 classified elements:")
for i, element in enumerate(classified_elements[:20]):
    print(f"Element {i+1}: {element}")

# Count the distribution of classifications
from collections import Counter
class_counts = Counter([e['classification'] for e in classified_elements])
print("\nClassification counts:")
print(class_counts)

First 20 classified elements:
Element 1: {'text': 'PostgreSQL 18.4 Documentation', 'page': 1, 'font_size': 24.883, 'font_name': 'Helvetica-Bold', 'bbox': (112.432, 693.562, 499.587, 718.445), 'classification': 'header'}
Element 2: {'text': 'The PostgreSQL Global Development Group', 'page': 1, 'font_size': 17.28, 'font_name': 'Helvetica-Bold', 'bbox': (124.051, 527.781, 487.968, 545.061), 'classification': 'header'}
Element 3: {'text': 'PostgreSQL 18.4 Documentation', 'page': 2, 'font_size': 14.4, 'font_name': 'Helvetica-Bold', 'bbox': (72.0, 704.7, 296.05, 719.1), 'classification': 'subheader'}
Element 4: {'text': 'The PostgreSQL Global Development Group', 'page': 2, 'font_size': 10.0, 'font_name': 'Times-Roman', 'bbox': (72.0, 692.22, 253.09, 702.22), 'classification': 'body_text'}
Element 5: {'text': 'Copyright © 1996–2026 The PostgreSQL Global Development Group', 'page': 2, 'font_size': 10.0, 'font_name': 'Times-Roman', 'bbox': (72.0, 680.22, 353.75, 690.22), 'classification': 'body

In [12]:
for i, element in enumerate(classified_elements[87065:87075]):
    print(f"Element {i+1}: {element}")

Element 1: {'text': '[ WITH [ RECURSIVE ] with_query [, ...] ]', 'page': 2166, 'font_size': 10.0, 'font_name': 'Courier', 'bbox': (120.0, 590.547, 366.0, 600.547), 'classification': 'body_text'}
Element 2: {'text': 'INSERT INTO table_name [ AS alias ] [ ( column_name [, ...] ) ]', 'page': 2166, 'font_size': 10.0, 'font_name': 'Courier', 'bbox': (120.0, 578.547, 498.0, 588.547), 'classification': 'query'}
Element 3: {'text': '[ OVERRIDING { SYSTEM | USER } VALUE ]', 'page': 2166, 'font_size': 10.0, 'font_name': 'Courier', 'bbox': (120.0, 566.547, 372.0, 576.547), 'classification': 'body_text'}
Element 4: {'text': '{ DEFAULT VALUES | VALUES ( { expression | DEFAULT } [, ...] )', 'page': 2166, 'font_size': 10.0, 'font_name': 'Courier', 'bbox': (120.0, 554.547, 516.0, 564.547), 'classification': 'body_text'}
Element 5: {'text': '[, ...] | query }', 'page': 2166, 'font_size': 10.0, 'font_name': 'Courier', 'bbox': (120.0, 542.547, 228.0, 552.547), 'classification': 'body_text'}
Element 6: {'

### Validating our classify_text_element function

#### Preparing Data for Decision Tree Model

To train a decision tree, we need to extract numerical features from our `Line` objects and use the current `classify_text_element` function to assign initial labels (our target variable). We'll also encode categorical features like `font_name`.

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.preprocessing import LabelEncoder
from io import StringIO
import pydotplus
from IPython.display import Image
def prepare_data(lines, classify_function):
  # Prepare features and labels from the 'lines' data
  data = []
  for line in lines:
      data.append({
          'text': line.text, # Used to see what the relevant text was
          'text_length': len(line.text),
          'font_size': line.font_size,
          'font_name': line.font_name,
          'bbox_x0': line.bbox[0],
          'bbox_y0': line.bbox[1],
          'bbox_x1': line.bbox[2],
          'bbox_y1': line.bbox[3],
          'page': line.page,
          'label': classify_function(line) # Use current heuristic as initial label
      })

  df = pd.DataFrame(data)

  # Encode categorical 'font_name'
  le = LabelEncoder()
  df['font_name_encoded'] = le.fit_transform(df['font_name'])

  # Define features (X) and target (y)
  X = df[['text_length', 'font_size', 'bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1', 'page', 'font_name_encoded']]
  y = df['label']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

  print(f"Original DataFrame shape: {df.shape}")
  print(f"Training set shape: {X_train.shape}")
  print(f"Testing set shape: {X_test.shape}")
  print("\nFeatures (first 5 rows of training data):")
  display(X_train.head())
  print("\nLabels (first 5 rows of training data):")
  display(y_train.head())
  return df, X_train, X_test, y_train, y_test

In [14]:
full_df, X_train, X_test, y_train, y_test = prepare_data(lines, classify_text_element)

Original DataFrame shape: (136970, 11)
Training set shape: (95879, 8)
Testing set shape: (41091, 8)

Features (first 5 rows of training data):


,text_length,font_size,bbox_x0,bbox_y0,bbox_x1,bbox_y1,page,font_name_encoded
76317,8,17.28,72.0,628.453,148.827,645.733,1833,5
94693,103,10.00,120.0,447.292,540.000,457.292,2382,9
134242,31,10.00,72.0,349.500,248.390,359.500,3282,9
24876,56,10.00,120.0,545.140,357.500,555.140,568,9
73063,131,10.00,120.0,661.500,539.999,671.500,1729,9



Labels (first 5 rows of training data):


,label
76317,header
94693,body_text
134242,body_text
24876,body_text
73063,body_text


#### Training a Decision Tree Classifier

Now, let's train a `DecisionTreeClassifier` using the prepared features and labels. We'll set a `max_depth` to avoid overfitting and keep the tree interpretable.

In [15]:
# Initialize and train the Decision Tree Classifier with max depth 2
dt_classifier_depth_2 = DecisionTreeClassifier(max_depth=2, random_state=42)
dt_classifier_depth_2.fit(X_train, y_train)

# Get training and testing accuracy from dt_classifier
training_accuracy_depth_2 = dt_classifier_depth_2.score(X_train, y_train)
testing_accuracy_depth_2 = dt_classifier_depth_2.score(X_test, y_test)

print(f"Training Accuracy: {training_accuracy_depth_2}")
print(f"Testing Accuracy: {testing_accuracy_depth_2}")

Training Accuracy: 0.9614514127181134
Testing Accuracy: 0.9614757489474581


In [16]:
# Initialize and train the Decision Tree Classifier with max depth 3
dt_classifier_depth_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_classifier_depth_3.fit(X_train, y_train)

# Get training and testing accuracy from dt_classifier
training_accuracy_depth_3 = dt_classifier_depth_3.score(X_train, y_train)
testing_accuracy_depth_3 = dt_classifier_depth_3.score(X_test, y_test)

print(f"Training Accuracy: {training_accuracy_depth_3}")
print(f"Testing Accuracy: {testing_accuracy_depth_3}")

Training Accuracy: 0.9643821900520447
Testing Accuracy: 0.9643474240101239


In [17]:
# Initialize and train the Decision Tree Classifier with max depth 4
dt_classifier_depth_4 = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_classifier_depth_4.fit(X_train, y_train)

# Get training and testing accuracy from dt_classifier
training_accuracy_depth_4 = dt_classifier_depth_4.score(X_train, y_train)
testing_accuracy_depth_4 = dt_classifier_depth_4.score(X_test, y_test)

print(f"Training Accuracy: {training_accuracy_depth_4}")
print(f"Testing Accuracy: {testing_accuracy_depth_4}")

Training Accuracy: 0.9644030496771973
Testing Accuracy: 0.9643960964688131


Here, it seems that a decision tree of depth 3 should be good enough to classify our text

#### Checking Errors in Decision Tree to Refine Our Classification Function

In [18]:
# Get rows where there are errors
errors_df = full_df[(full_df['label'] != dt_classifier_depth_3.predict(full_df[['text_length', 'font_size', 'bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1', 'page', 'font_name_encoded']]))].copy()
# Predict only for the rows that are actually in errors_df
errors_df['predicted_label'] = dt_classifier_depth_3.predict(errors_df[['text_length', 'font_size', 'bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1', 'page', 'font_name_encoded']])

# Display the first few rows of errors
print(errors_df.head(50))

                                                   text  text_length  \
2114  ----------------------------------------------...           67   
2115                            -----------------------           23   
2121                                       ------------           12   
2126                                         ----------           10   
2178                             CREATE TABLE weather (           22   
2201                              CREATE TABLE cities (           21   
2208                              DROP TABLE tablename;           21   
2211  INSERT INTO weather VALUES ('San Francisco', 4...           58   
2219  INSERT INTO cities VALUES ('San Francisco', '(...           62   
2222  INSERT INTO weather (city, temp_lo, temp_hi, p...           56   
2226  INSERT INTO weather (date, city, temp_hi, temp...           50   
2233        COPY weather FROM '/home/user/weather.txt';           43   
2246                             SELECT * FROM weather;         

In [19]:
print(errors_df.tail(50))

                                                     text  text_length  \
131059  INSERT INTO connectby_tree VALUES('row9','row5...           52   
131062  SELECT * FROM connectby('connectby_tree', 'key...           66   
131076  SELECT * FROM connectby('connectby_tree', 'key...           66   
131089  SELECT * FROM connectby('connectby_tree', 'key...           66   
131103  SELECT * FROM connectby('connectby_tree', 'key...           66   
131144                                       CREATE TABLE           12   
131149                                     CREATE TRIGGER           14   
131151                                             LISTEN            6   
131158                                    with PID 22770.           15   
131161                                    with PID 22770.           15   
131164                                    with PID 22770.           15   
131166                                           UPDATE 2            8   
131172                                

In [20]:
errors_thought_output_predicted_body = errors_df[(errors_df['label'] == 'output') & (errors_df['predicted_label'] == 'body_text')]
errors_thought_query_predicted_body = errors_df[(errors_df['label'] == 'query') & (errors_df['predicted_label'] == 'body_text')]
errors_thought_note_predicted_body = errors_df[(errors_df['label'] == 'note_marker') & (errors_df['predicted_label'] == 'body_text')]
errors_thought_subheader_predicted_note = errors_df[(errors_df['label'] == 'subheader') & (errors_df['predicted_label'] == 'note_marker')]
print(f'Number of errors thought to be output and predicted body: {len(errors_thought_output_predicted_body)}')
print(f'Number of errors thought to be query and predicted body: {len(errors_thought_query_predicted_body)}')
print(f'Number of errors thought to be note and predicted subheader: {len(errors_thought_note_predicted_body)}')
print(f'Number of errors thought to be subheader and predicted note: {len(errors_thought_subheader_predicted_note)}')
print(f'Number of errors total: {len(errors_df)}')

Number of errors thought to be output and predicted body: 548
Number of errors thought to be query and predicted body: 4327
Number of errors thought to be note and predicted subheader: 1
Number of errors thought to be subheader and predicted note: 4
Number of errors total: 4880


So here, almost all of our errors come from our DT classifier predicting as body text and it being either output or query. I checked the lines in the documentation and my classify_text_element seems to get all of them correct, as well as the other types of errors given, so I will use that

In [21]:
for i, line in enumerate(lines):
  line.classification = classify_text_element(line, all_elements=lines, current_index=i)

## Grouping Lines into Blocks

Now, let's group the individual `Line` objects into larger `Block` structures. This process will primarily rely on the vertical spacing between consecutive lines and their page numbers.

**Logic for Block Formation:**
1.  A new block is started if the current line is on a different page than the previous one.
2.  A new block is started if the vertical distance between the bottom of the previous line and the top of the current line exceeds a certain threshold (e.g., 1.5 times the `body_font_size`). This accounts for natural paragraph breaks or larger structural separations.
3.  All lines within a continuous sequence that meet these criteria will be grouped into a single `Block` object. The `block_type`, `header`, and `section` attributes will be left as `None` for now, as per the request to separate lines without classification.

In [22]:
@dataclass
class Block:
    lines: list[Line]
    pages: list[int]
    block_type: str
    header: str | None
    section: str | None

In [23]:
def group_lines_into_blocks(all_lines, vertical_threshold=1, filter_footer_y0_threshold=70, filter_header_y3_threshold=720):
    blocks = []
    current_block_lines = []

    # Filter out lines that are likely page numbers/footers or headers

    filtered_lines = [line for line in all_lines if line.bbox[1] > filter_footer_y0_threshold and line.bbox[3] < filter_header_y3_threshold]
    filtered_lines = sorted(
        filtered_lines,
        key=lambda l: (
            l.page,
            -l.bbox[3],
            l.bbox[0]
        )
    )

    if not filtered_lines:
        return blocks

    for i in range(1, len(filtered_lines)):
        prev_line = filtered_lines[i-1]
        current_line = filtered_lines[i]

        should_start_new_block = False

        # Rule 1: Always start a new block if the current line is a header, subheader, or note_marker.
        # This ensures these elements always begin a fresh block.
        if current_line.classification in ['header', 'subheader', 'note_marker']:
            should_start_new_block = True
        # Rule 2: If the previous line was a note_marker, the current line should generally be grouped with it,
        # unless the current line itself is a structural element (header/subheader) or
        # there's an exceptionally large vertical gap indicating the end of the note content.
        elif prev_line.classification == 'note_marker':
            # If the current line is a header/subheader, it signifies the end of the note block.
            # This is important to ensure a note's content doesn't bleed into a new section's header.
            if current_line.classification in ['header', 'subheader']:
                should_start_new_block = True
            # If the vertical gap is very large (e.g., more than twice the normal threshold),
            # it also signifies the end of the note block, even if the content is not a header.
            elif (prev_line.bbox[1] - current_line.bbox[3] > body_font_size * vertical_threshold * 4): # Increased threshold from *2 to *4
                should_start_new_block = True
            # Otherwise (current line is not a structural element and gap is not too large),
            # the current line is considered part of the note's content, so do NOT start a new block.
            else:
                should_start_new_block = False
        # Rule 3: For all other cases (e.g., body_text following body_text, query following query, etc.),
        # use the standard vertical spacing to determine if a new block should start.
        elif (prev_line.bbox[1] - current_line.bbox[3] > body_font_size * vertical_threshold):
            should_start_new_block = True
        # Default to not starting a new block if no conditions above are met.
        else:
            should_start_new_block = False

        if should_start_new_block:
            blocks.append(Block(
                lines=current_block_lines,
                pages=sorted(list(set(l.page for l in current_block_lines))), # Use sorted for consistency
                block_type=None,
                header=None,
                section=None
            ))
            current_block_lines = [current_line]
            continue
        else:
            current_block_lines.append(current_line)

    # Add the last block if it's not empty
    if current_block_lines:
        blocks.append(Block(
            lines=current_block_lines,
            pages=sorted(list(set(l.page for l in current_block_lines))), # Use sorted for consistency
            block_type=None,
            header=None,
            section=None
        ))
    return blocks

In [24]:
# Group the lines into blocks
block_elements = group_lines_into_blocks(lines, vertical_threshold=0.5)

print(f"Extracted {len(block_elements)} blocks.")

# Display some information about the blocks whose starting page is in the range [38, 41]
print("\nBlocks with starting page in range [38, 41]:")
for i, block in enumerate(block_elements):
    if block.pages and 38 <= block.pages[0] <= 41:
        print(f"Block {i+1}: Type={block.block_type}, Lines={len(block.lines)}, Pages = {block.pages}")
        if block.lines:
            print(f"  First Line: '{block.lines[0].text}'")
            print(f"  Last Line: '{block.lines[-1].text}'")
        print("--------------------------------------------------")

Extracted 45912 blocks.

Blocks with starting page in range [38, 41]:
Block 81: Type=None, Lines=1, Pages = [38]
  First Line: '• A program produces the wrong output for any given input.'
  Last Line: '• A program produces the wrong output for any given input.'
--------------------------------------------------
Block 82: Type=None, Lines=1, Pages = [38]
  First Line: '• A program refuses to accept valid input (as defined in the documentation).'
  Last Line: '• A program refuses to accept valid input (as defined in the documentation).'
--------------------------------------------------
Block 83: Type=None, Lines=2, Pages = [38]
  First Line: '• A program accepts invalid input without a notice or error message. But keep in mind that your idea of'
  Last Line: 'invalid input might be our idea of an extension or compatibility with traditional practice.'
--------------------------------------------------
Block 84: Type=None, Lines=1, Pages = [38]
  First Line: '• PostgreSQL fails to compile

### Implementing `classify_block_element`

To classify blocks, we'll use a set of heuristics that consider the overall characteristics of the lines within each block:

1.  **Header/Subheader Blocks:** If any line within a block is classified as a 'header' or 'subheader' by `classify_text_element`, the block is classified as such. This captures titles and section headings.
2.  **Query Blocks:** If a significant portion (e.g., 50% or more) of the lines in a block are classified as 'query' by `classify_text_element`, and it's not already a header/subheader, it's considered a 'query_block'. This targets SQL code examples.
3.  **Output Blocks:** Similarly, if a significant portion of lines are 'output', it becomes an 'output_block'. This covers tables and program output.
4.  **Body Text Blocks:** If none of the above rules apply, the block defaults to 'body_block', representing general explanatory text.
5. **First Line Font Size/Name:** We can also incorporate the font size and name of the first line as a strong indicator for block type, especially for headers/subheaders or consistent code blocks.

In [25]:
def classify_block_element(block_element):
    # Ensure block_element.lines is not empty
    if not block_element.lines:
        return 'empty_block'

    # Get classifications for all lines within the block using the existing line classifier
    line_classifications = []
    for i, line in enumerate(block_element.lines):
        # We need to pass all_elements and current_index for context-aware classification
        # For block classification, we are considering the lines *within* the block as a sequence.
        # This might slightly differ from the global classification context, but it's a good start.
        line_classifications.append(line.classification)

    # Heuristic 1: If any line is a 'header' or 'subheader', classify the block accordingly
    if 'header' in line_classifications:
        return 'header_block'
    if 'subheader' in line_classifications:
        return 'subheader_block'

    # Heuristic: If any line is a 'note_marker', classify the block as 'note_block'
    if 'note_marker' in line_classifications:
        return 'note_block'

    # Count occurrences of different classifications within the block
    from collections import Counter
    class_counts = Counter(line_classifications)

    total_lines = len(line_classifications)

    # Heuristic 2: Check for query blocks (e.g., if a majority of lines are queries)
    query_percentage = class_counts['query'] / total_lines
    if query_percentage >= 0.5: # More than 50% of lines are queries
        return 'query_block'

    # Heuristic 3: Check for output blocks (e.g., if a majority of lines are output)
    output_percentage = class_counts['output'] / total_lines
    if output_percentage >= 0.5: # More than 50% of lines are output
        return 'output_block'

    # Heuristic 4: Use first line's font characteristics as a strong indicator for some cases
    first_line = block_element.lines[0]
    if first_line.font_name == 'Courier' and first_line.font_size == body_font_size and first_line.bbox[0] > 110: # Example of code/query-like formatting
        # If it's Courier and looks like a code block but not majority query/output, it's likely a body text code snippet
        # or a query that didn't meet the 50% threshold (e.g. short queries or single lines).
        # Given the previous `classify_text_element` logic, Courier lines often imply query/output, even if not definitive SQL.
        if 'query' in line_classifications or 'output' in line_classifications:
          # If there's any query or output in a Courier block, it's probably a code/output block
          if class_counts['query'] > class_counts['output']:
            return 'query_block'
          elif class_counts['output'] > 0:
            return 'output_block'
          else:
            # If it has courier text elements and is neither a query nor an output based on counts, it should still be a query block, as the font implies.
            return 'query_block'

    # Default to body_block if no other specific classification applies
    return 'body_block'

In [26]:
# Apply the classify_block_element function to all block_elements
for block in block_elements:
    block.block_type = classify_block_element(block)

block_elements = [block for block in block_elements if block.block_type != 'empty_block'] # Remove empty blocks

print(f"Classified {len(block_elements)} blocks.")

# Display the first 10 classified blocks with their types
print("\nFirst 10 classified blocks:")
for i, block in enumerate(block_elements[:10]):
    print(f"Block {i+1}: Type={block.block_type}, Lines={len(block.lines)}, Pages={block.pages}")
    if block.lines:
        print(f"  First Line: '{block.lines[0].text}'")
        print(f"  Last Line: '{block.lines[-1].text}'")
    print("--------------------------------------------------")

# Count the distribution of block types
from collections import Counter
block_type_counts = Counter([b.block_type for b in block_elements])
print("\nBlock Type Counts:")
print(block_type_counts)

Classified 45911 blocks.

First 10 classified blocks:
Block 1: Type=header_block, Lines=1, Pages=[1]
  First Line: 'The PostgreSQL Global Development Group'
  Last Line: 'The PostgreSQL Global Development Group'
--------------------------------------------------
Block 2: Type=subheader_block, Lines=3, Pages=[2]
  First Line: 'PostgreSQL 18.4 Documentation'
  Last Line: 'Copyright © 1996–2026 The PostgreSQL Global Development Group'
--------------------------------------------------
Block 3: Type=subheader_block, Lines=1, Pages=[2]
  First Line: 'Legal Notice'
  Last Line: 'Legal Notice'
--------------------------------------------------
Block 4: Type=body_block, Lines=1, Pages=[2]
  First Line: 'PostgreSQL Database Management System (also known as Postgres, formerly known as Postgres95)'
  Last Line: 'PostgreSQL Database Management System (also known as Postgres, formerly known as Postgres95)'
--------------------------------------------------
Block 5: Type=body_block, Lines=1, Pages=[

### Grouping Content Hierarchically for RAG

To prepare the data for a Retrieval Augmented Generation (RAG) application, it's beneficial to group related content under its respective header and subheader. This creates logically coherent chunks of information that can be retrieved more effectively.

We will define a new structure, `HierarchicalBlock`, which will contain a `header_block` (if available), a `subheader_block` (if available), and a list of `content_blocks` (body, query, output, note blocks) that fall under that hierarchy.


In [27]:
from dataclasses import dataclass, field

@dataclass
class HierarchicalBlock:
    header_block: Block | None = None
    subheader_block: Block | None = None
    content_blocks: list[Block] = field(default_factory=list)
    block_type: str = "hierarchical_block"


def group_into_hierarchical_blocks(blocks):
    hierarchical_blocks = []
    current_header = None
    current_subheader = None
    current_hierarchical_block = None

    for block in blocks:
        if block.block_type == 'header_block':
            # If a new header is encountered, close the previous hierarchical block
            # and start a new one.
            if current_hierarchical_block and (current_hierarchical_block.content_blocks or current_hierarchical_block.header_block or current_hierarchical_block.subheader_block):
                hierarchical_blocks.append(current_hierarchical_block)

            current_header = block
            current_subheader = None # Reset subheader when a new header starts
            current_hierarchical_block = HierarchicalBlock(header_block=current_header)

        elif block.block_type == 'subheader_block':
            # If a new subheader is encountered, close the previous hierarchical block
            # and start a new one, inheriting the current header.
            if current_hierarchical_block and (current_hierarchical_block.content_blocks or current_hierarchical_block.header_block or current_hierarchical_block.subheader_block):
                hierarchical_blocks.append(current_hierarchical_block)

            current_subheader = block
            current_hierarchical_block = HierarchicalBlock(header_block=current_header, subheader_block=current_subheader)

        else: # Content blocks (body, query, output, note)
            if current_hierarchical_block is None:
                # Handle content blocks before any header/subheader is found
                current_hierarchical_block = HierarchicalBlock()
            current_hierarchical_block.content_blocks.append(block)

    # Add the last hierarchical block if it exists and contains content
    if current_hierarchical_block and (current_hierarchical_block.content_blocks or current_hierarchical_block.header_block or current_hierarchical_block.subheader_block):
        hierarchical_blocks.append(current_hierarchical_block)

    return hierarchical_blocks

hierarchical_elements = group_into_hierarchical_blocks(block_elements)

print(f"Created {len(hierarchical_elements)} hierarchical blocks.")

# Display the first few hierarchical blocks to verify the structure
for i, h_block in enumerate(hierarchical_elements[:5]):
    print(f"\nHierarchical Block {i+1}:")
    if h_block.header_block:
        print(f"  Header: '{h_block.header_block.lines[0].text}' (Page: {h_block.header_block.pages[0]}) (Type: {h_block.header_block.block_type})")
    if h_block.subheader_block:
        print(f"  Subheader: '{h_block.subheader_block.lines[0].text}' (Page: {h_block.subheader_block.pages[0]}) (Type: {h_block.subheader_block.block_type})")
    print(f"  Content Blocks Count: {len(h_block.content_blocks)}")
    for j, content_b in enumerate(h_block.content_blocks[:3]): # Show first 3 content blocks for brevity
        print(f"    Content Block {j+1}: Type={content_b.block_type}, Lines={len(content_b.lines)}, Pages={content_b.pages}, First Line='{content_b.lines[0].text}', Last Line='{content_b.lines[-1].text}'")
    if len(h_block.content_blocks) > 3:
        print(f"    ... and {len(h_block.content_blocks) - 3} more content blocks")
    if h_block.content_blocks: # Add this check
        last_content_block = h_block.content_blocks[-1]
        print(f" Last Content Block ({len(h_block.content_blocks)}): Type={last_content_block.block_type}, Lines={len(last_content_block.lines)}, Pages={last_content_block.pages}, First Line='{last_content_block.lines[0].text}', Last Line='{last_content_block.lines[-1].text}'")


Created 5132 hierarchical blocks.

Hierarchical Block 1:
  Header: 'The PostgreSQL Global Development Group' (Page: 1) (Type: header_block)
  Content Blocks Count: 0

Hierarchical Block 2:
  Header: 'The PostgreSQL Global Development Group' (Page: 1) (Type: header_block)
  Subheader: 'PostgreSQL 18.4 Documentation' (Page: 2) (Type: subheader_block)
  Content Blocks Count: 0

Hierarchical Block 3:
  Header: 'The PostgreSQL Global Development Group' (Page: 1) (Type: header_block)
  Subheader: 'Legal Notice' (Page: 2) (Type: subheader_block)
  Content Blocks Count: 6
    Content Block 1: Type=body_block, Lines=1, Pages=[2], First Line='PostgreSQL Database Management System (also known as Postgres, formerly known as Postgres95)', Last Line='PostgreSQL Database Management System (also known as Postgres, formerly known as Postgres95)'
    Content Block 2: Type=body_block, Lines=1, Pages=[2], First Line='Portions Copyright © 1996-2026, PostgreSQL Global Development Group', Last Line='Portions

In [28]:
for i, h_block in enumerate(hierarchical_elements[2130:2175]):
    print(f"\nHierarchical Block {i+1}:")
    if h_block.header_block:
        print(f"  Header: '{h_block.header_block.lines[0].text}' (Page: {h_block.header_block.pages[0]}) (Type: {h_block.header_block.block_type})")
    if h_block.subheader_block:
        print(f"  Subheader: '{h_block.subheader_block.lines[0].text}' (Page: {h_block.subheader_block.pages[0]}) (Type: {h_block.subheader_block.block_type})")
    print(f"  Content Blocks Count: {len(h_block.content_blocks)}")
    for j, content_b in enumerate(h_block.content_blocks[:3]): # Show first 3 content blocks for brevity
        print(f"    Content Block {j+1}: Type={content_b.block_type}, Lines={len(content_b.lines)}, Pages={content_b.pages}, First Line='{content_b.lines[0].text}', Last Line='{content_b.lines[-1].text}'")
    if len(h_block.content_blocks) > 3:
        print(f"    ... and {len(h_block.content_blocks) - 3} more content blocks")
    if h_block.content_blocks: # Add this check
        last_content_block = h_block.content_blocks[-1]
        print(f" Last Content Block ({len(h_block.content_blocks)}): Type={last_content_block.block_type}, Lines={len(last_content_block.lines)}, Pages={last_content_block.pages}, First Line='{last_content_block.lines[0].text}', Last Line='{last_content_block.lines[-1].text}'")



Hierarchical Block 1:
  Header: 'Synopsis' (Page: 1671) (Type: header_block)
  Subheader: 'Description' (Page: 1671) (Type: subheader_block)
  Content Blocks Count: 2
    Content Block 1: Type=body_block, Lines=1, Pages=[1671], First Line='SPI_palloc allocates memory in the upper executor context.', Last Line='SPI_palloc allocates memory in the upper executor context.'
    Content Block 2: Type=body_block, Lines=1, Pages=[1671], First Line='This function can only be used while connected to SPI. Otherwise, it throws an error.', Last Line='This function can only be used while connected to SPI. Otherwise, it throws an error.'
 Last Content Block (2): Type=body_block, Lines=1, Pages=[1671], First Line='This function can only be used while connected to SPI. Otherwise, it throws an error.', Last Line='This function can only be used while connected to SPI. Otherwise, it throws an error.'

Hierarchical Block 2:
  Header: 'Synopsis' (Page: 1671) (Type: header_block)
  Subheader: 'Arguments' (P

This block function seems to work very well

## Chunking Hierarchical Blocks

Before we can create embeddings, we need to break down our `HierarchicalBlock` objects into smaller, fixed-size text chunks. This is crucial for effective retrieval, as very large documents might dilute the relevance of embeddings, and very small ones might lack context. We'll aim for chunks of a specific size with some overlap to maintain continuity.

Each `TextChunk` will contain:
- The actual `text` content.
- The `page_numbers` it covers.
- The `source_header` and `source_subheader` for context.
- A unique `chunk_id` for identification.

In [29]:
import uuid

@dataclass
class TextChunk:
    chunk_id: str
    text: str
    page_numbers: list[int]
    source_header: str | None = None
    source_subheader: str |None = None
    block_type: str | None = None
    embedding = None


def chunk_hierarchical_blocks(
        h_blocks,
        chunk_size=1200,
        chunk_overlap=200):

    chunks = []

    for h_block in h_blocks:

        pages = set()

        header = None
        subheader = None

        if h_block.header_block:
            header = h_block.header_block.lines[0].text
            pages.update(h_block.header_block.pages)

        if h_block.subheader_block:
            subheader = h_block.subheader_block.lines[0].text
            pages.update(h_block.subheader_block.pages)

        current_chunk = []
        current_length = 0

        paragraphs = []

        paragraph_types = []

        # Each content block becomes one paragraph
        for block in h_block.content_blocks:

            paragraph = "\n".join(line.text for line in block.lines)

            if paragraph.strip():
                paragraphs.append(paragraph)
                paragraph_types.append(block.block_type)

            pages.update(block.pages)

        for paragraph, block_type in zip(paragraphs, paragraph_types):

            paragraph = paragraph.strip()

            if not paragraph:
                continue

            paragraph_length = len(paragraph)

            # Large paragraph -> split only this paragraph
            if paragraph_length > chunk_size:

                # flush existing chunk
                if current_chunk:
                    chunks.append(
                        TextChunk(
                            chunk_id=str(uuid.uuid4()),
                            text="\n\n".join(current_chunk),
                            page_numbers=sorted(pages),
                            source_header=header,
                            source_subheader=subheader,
                            block_type=block_type
                        )
                    )

                    current_chunk = []
                    current_length = 0

                start = 0

                while start < paragraph_length:

                    end = min(start + chunk_size, paragraph_length)

                    # try to end at whitespace
                    if end < paragraph_length:
                        while end > start and paragraph[end] not in (" ", "\n"):
                            end -= 1

                    if end == start:
                        end = min(start + chunk_size, paragraph_length)

                    piece = paragraph[start:end].strip()

                    chunks.append(
                        TextChunk(
                            chunk_id=str(uuid.uuid4()),
                            text=piece,
                            page_numbers=sorted(pages),
                            source_header=header,
                            source_subheader=subheader,
                            block_type=block_type
                        )
                    )

                    start = max(end - chunk_overlap, end)

                continue

            # Doesn't fit current chunk -> emit current chunk
            if current_length + paragraph_length > chunk_size and current_chunk:

                chunks.append(
                    TextChunk(
                        chunk_id=str(uuid.uuid4()),
                        text="\n\n".join(current_chunk),
                        page_numbers=sorted(pages),
                        source_header=header,
                        source_subheader=subheader,
                        block_type=block_type
                    )
                )

                # paragraph overlap
                overlap = []

                overlap_len = 0

                for p in reversed(current_chunk):

                    if overlap_len + len(p) > chunk_overlap:
                        break

                    overlap.insert(0, p)
                    overlap_len += len(p)

                current_chunk = overlap
                current_length = sum(len(x) for x in overlap)

            current_chunk.append(paragraph)
            current_length += paragraph_length

        if current_chunk:

            chunks.append(
                TextChunk(
                    chunk_id=str(uuid.uuid4()),
                    text="\n\n".join(current_chunk),
                    page_numbers=sorted(pages),
                    source_header=header,
                    source_subheader=subheader,
                    block_type=paragraph_types[0] if paragraph_types else None
                )
            )

    return chunks

# Perform chunking with the updated function
final_text_chunks = chunk_hierarchical_blocks(hierarchical_elements, chunk_size=700, chunk_overlap=100)

print(f"Generated {len(final_text_chunks)} text chunks.")

# Display the first few chunks
for i, chunk in enumerate(final_text_chunks[:5]):
    print(f"\nChunk {i+1}:")
    print(f"\tHeader: {chunk.source_header}")
    print(f"\tSubheader: {chunk.source_subheader}")
    print(f"\tBlock Type: {chunk.block_type}")
    print(f"\tPages: {chunk.page_numbers}")
    print(f"\tText (first 200 chars): {chunk.text[:200]}...")

Generated 15450 text chunks.

Chunk 1:
	Header: The PostgreSQL Global Development Group
	Subheader: Legal Notice
	Block Type: body_block
	Pages: [1, 2]
	Text (first 200 chars): PostgreSQL Database Management System (also known as Postgres, formerly known as Postgres95)

Portions Copyright © 1996-2026, PostgreSQL Global Development Group

Portions Copyright © 1994, The Regent...

Chunk 2:
	Header: The PostgreSQL Global Development Group
	Subheader: Legal Notice
	Block Type: body_block
	Pages: [1, 2]
	Text (first 200 chars): IN NO EVENT SHALL THE UNIVERSITY OF CALIFORNIA BE LIABLE TO ANY PARTY FOR DIRECT, INDIRECT, SPECIAL, INCI-
DENTAL, OR CONSEQUENTIAL DAMAGES, INCLUDING LOST PROFITS, ARISING OUT OF THE USE OF THIS SOFT...

Chunk 3:
	Header: Table of Contents
	Subheader: None
	Block Type: body_block
	Pages: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
	Text (first 200 chars): Preface ..........................................................................

In [30]:
chunk_lengths = [len(chunk.text) for chunk in final_text_chunks]

print(min(chunk_lengths))
print(max(chunk_lengths))
print(sum(chunk_lengths)/len(chunk_lengths))

1
801
475.2394822006473


In [31]:
for i, chunk in enumerate(final_text_chunks[11000:11005]):
  print(f"chunk {i+1} text: {chunk.text}")
  print("\n\n")

chunk 1 text: alias

A substitute name for the target table. When an alias is provided, it completely hides the actual name
of the table. For example, given UPDATE foo AS f, the remainder of the UPDATE statement must
refer to this table as f not foo.

column_name

The name of a column in the table named by table_name. The column name can be qualified with
a subfield name or array subscript, if needed. Do not include the table's name in the specification of a
target column — for example, UPDATE table_name SET table_name.col = 1 is invalid.

expression

An expression to assign to the column. The expression can use the old values of this and other columns
in the table.

DEFAULT



chunk 2 text: DEFAULT

Set the column to its default value (which will be NULL if no specific default expression has been
assigned to it). An identity column will be set to a new value generated by the associated sequence.
For  a  generated  column,  specifying  this  is  permitted  but  merely  specifies  the  

In [32]:
# Statistics on chunks
# Number of chunks
print(f"Total number of chunks: {len(final_text_chunks)}")
# Average length of chunks
print(f"Average length of chunks: {sum(len(chunk.text) for chunk in final_text_chunks) / len(final_text_chunks)}")

Total number of chunks: 15450
Average length of chunks: 475.2394822006473


## Generate Embeddings for Text Chunks

Now that we have our `TextChunk` objects, we need to convert them into numerical vector embeddings. These embeddings will be used to measure the semantic similarity between a user's query and the documentation chunks, allowing for efficient retrieval of relevant information.

We'll use the `GoogleGenerativeAI` embedding model (`text-embedding-004`), which is suitable for general-purpose text embedding. Each `TextChunk` will have an `embedding` attribute added, storing its vector representation.

## Storing Embeddings in a PostgreSQL Vector Database (pgvector)

In [33]:
import os

# Load your HuggingFace Token from Colab secrets
HF_TOKEN = userdata.get('HF_READ_TOKEN')

# Set the environment variable for HuggingFace
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN # Corrected to HF_TOKEN
    print("HuggingFace token successfully loaded and configured.")
else:
    print("HuggingFace token not found in Colab secrets. Proceeding with unauthenticated requests.")

HuggingFace token successfully loaded and configured.


In [34]:
from sentence_transformers import SentenceTransformer

embedding_model_name = SentenceTransformer("BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [35]:
from tqdm.auto import tqdm # Import tqdm for progress bar

def get_embedding(text):
    try:
        # This function is now primarily for single text embeddings, e.g., for user queries.
        embedding = embedding_model_name.encode(text)
        return embedding.tolist()
    except Exception as e:
        print(f"Error getting embedding for text: {text[:50]}...")
        print(f"Error: {e}")
        return None

final_text_chunks_with_embeddings = []
batch_size = 64 # Define a batch size for embedding generation. Adjust based on GPU memory.

print(f"Generating embeddings for {len(final_text_chunks)} chunks in batches of {batch_size}...")

# Use tqdm for a progress bar
for i in tqdm(range(0, len(final_text_chunks), batch_size), desc="Generating Embeddings"):
    batch_chunks = final_text_chunks[i:i + batch_size]
    batch_texts = [chunk.text for chunk in batch_chunks]

    try:
        # Encode the batch of texts. convert_to_numpy=False, convert_to_tensor=False to get Python lists.
        batch_embeddings = embedding_model_name.encode(batch_texts, convert_to_numpy=False, convert_to_tensor=False)
        for j, chunk in enumerate(batch_chunks):
            chunk.embedding = batch_embeddings[j].tolist()
            final_text_chunks_with_embeddings.append(chunk)
    except Exception as e:
        print(f"Error getting embeddings for batch starting at index {i}. Attempting individual processing. Error: {e}")
        # Fallback: if batch encoding fails, try individual chunks
        for chunk in batch_chunks:
            embedding = get_embedding(chunk.text) # Use the single embedding function for robustness
            if embedding is not None:
                chunk.embedding = embedding
                final_text_chunks_with_embeddings.append(chunk)

print(f"Generated embeddings for {len(final_text_chunks_with_embeddings)} out of {len(final_text_chunks)} chunks.")

# Display the first chunk with its embedding (first 5 values)
if final_text_chunks_with_embeddings:
    first_chunk = final_text_chunks_with_embeddings[0]
    print(f"\nFirst chunk text: {first_chunk.text[:100]}...")
    print(f"First chunk embedding (first 5 values): {first_chunk.embedding[:5]}")
    print(f"Embedding dimension: {len(first_chunk.embedding)}")
    print(f"Embedding type: {type(first_chunk.embedding[0])}")
    EMBEDDING_DIMENSION = len(first_chunk.embedding)
else:
    EMBEDDING_DIMENSION = 0 # Default if no chunks are processed

Generating embeddings for 15450 chunks in batches of 64...


Generating Embeddings:   0%|          | 0/242 [00:00<?, ?it/s]

Generated embeddings for 15450 out of 15450 chunks.

First chunk text: PostgreSQL Database Management System (also known as Postgres, formerly known as Postgres95)

Portio...
First chunk embedding (first 5 values): [-0.032539788633584976, -0.08195803314447403, -0.009497181512415409, -0.05684235319495201, 0.0224609412252903]
Embedding dimension: 384
Embedding type: <class 'float'>


In [36]:
# Install the PostgreSQL adapter for Python
!pip install psycopg2-binary
!pip install pgvector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 119.6 MB/s eta 0:00:00


In [37]:
import psycopg2
from pgvector.psycopg2 import register_vector
from pgvector import Vector # Corrected import for Vector class
import os

# --- Database Connection Details ---
PG_HOST = userdata.get('PGSQL_AI_CHATBOT_NEONDB_HOST')
PG_PORT = 5432
PG_USER = 'neondb_owner'
PG_PASSWORD = userdata.get('PGSQL_AI_CHATBOT_NEONDB_PASSWORD')
PG_DBNAME = 'neondb'

def get_db_connection():
    conn = psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        user=PG_USER,
        password=PG_PASSWORD,
        dbname=PG_DBNAME,
        sslmode='require' # Add this line to enforce SSL
    )
    register_vector(conn) # Register the pgvector type with psycopg2
    return conn

print("Database connection parameters set. Ensure you've replaced placeholders or configured Colab secrets.")

Database connection parameters set. Ensure you've replaced placeholders or configured Colab secrets.


### Create Table for Text Chunks and Embeddings

We'll create a table to store each `TextChunk`'s data along with its `embedding`. The `embedding` column will use the `VECTOR` type provided by `pgvector`, with a dimension matching our embedding model (1536 for `text-embedding-004`).

In [38]:
def create_chunks_table():
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()

        # Enable pgvector extension if not already enabled
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

        # Create the table to store text chunks and their embeddings
        # We'll include metadata for context and source information
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS document_chunks (
                chunk_id VARCHAR(255) PRIMARY KEY,
                text TEXT NOT NULL,
                page_numbers INTEGER[],
                source_header TEXT,
                source_subheader TEXT,
                block_type VARCHAR(50),
                embedding VECTOR({EMBEDDING_DIMENSION})
                );
        """)

        conn.commit()
        print("Table 'document_chunks' created or already exists.")

        # Re-enabling HNSW index creation as the dimension (1536) is now within limits.
        cur.execute(f"CREATE INDEX IF NOT EXISTS idx_document_chunks_embedding ON document_chunks USING hnsw (embedding vector_l2_ops) WITH (m = 16, ef_construction = 64);")
        conn.commit()
        print("HNSW index created or already exists on 'document_chunks.embedding'.")

    except Exception as e:
        print(f"Error creating table or index: {e}")
    finally:
        if conn:
            conn.close()

create_chunks_table()

Table 'document_chunks' created or already exists.
HNSW index created or already exists on 'document_chunks.embedding'.


### Insert Text Chunks and Embeddings into PostgreSQL

Now we'll iterate through our `final_text_chunks_with_embeddings` list and insert each chunk's data, including its embedding, into the `document_chunks` table.

In [39]:
def insert_chunks_to_db(chunks):
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()

        # Prepare the INSERT statement
        insert_sql = """
            INSERT INTO document_chunks (chunk_id, text, page_numbers, source_header, source_subheader, block_type, embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (chunk_id) DO UPDATE SET
                text = EXCLUDED.text,
                page_numbers = EXCLUDED.page_numbers,
                source_header = EXCLUDED.source_header,
                source_subheader = EXCLUDED.source_subheader,
                block_type = EXCLUDED.block_type,
                embedding = EXCLUDED.embedding;
        """
        # This handles re-running the cell and updating existing chunks by chunk_id

        batch_size = 300 # Adjust batch size based on your system's performance
        for i in range(0, len(chunks), batch_size):
          # TODO set start back to 0
            batch = chunks[i:i+batch_size]
            data_to_insert = []
            for chunk in batch:
                # Convert page_numbers list to a PostgreSQL array literal string
                page_numbers_array = '{' + ','.join(map(str, chunk.page_numbers)) + '}' if chunk.page_numbers else '{}'
                data_to_insert.append((
                    chunk.chunk_id,
                    chunk.text,
                    page_numbers_array, # Pass as string
                    chunk.source_header,
                    chunk.source_subheader,
                    chunk.block_type,
                    chunk.embedding # pgvector's register_vector handles this type,
                ))
            cur.executemany(insert_sql, data_to_insert)
            conn.commit()
            print(f"Inserted/Updated {min(i + batch_size, len(chunks))}/{len(chunks)} chunks.")

        print(f"Successfully inserted/updated {len(chunks)} chunks into 'document_chunks' table.")

    except Exception as e:
        print(f"Error inserting chunks: {e}")
        if conn:
            conn.rollback() # Rollback in case of error
    finally:
        if conn:
            conn.close()

#insert_chunks_to_db(final_text_chunks_with_embeddings)
# Run this when uploading chunks to DB
# but note that it can take more than an hour to run

In [40]:
def add_search_vector():
  try:
    conn = get_db_connection()
    cur = conn.cursor()

    # Add search_vector column
    add_search_vector_col_query = """
        ALTER TABLE document_chunks
        ADD COLUMN IF NOT EXISTS search_vector tsvector
    """
    cur.execute(add_search_vector_col_query)
    conn.commit()
    print("search_vector column added or already exists.")
  except Exception as e:
    print(f"Error adding search_vector column: {e}")

  # Populate search_vector column
  try:
    conn = get_db_connection()
    cur = conn.cursor()
    update_search_vector_query = """
        UPDATE document_chunks
        SET search_vector = to_tsvector('english', text)
    """
    cur.execute(update_search_vector_query)
    conn.commit()
  except Exception as e:
    print(f"Error updating search_vector column: {e}")

  # Create the GIN index on the search vector column
  try:
    conn = get_db_connection()
    cur = conn.cursor()
    create_search_vector_index_query = """
        CREATE INDEX IF NOT EXISTS idx_document_chunks_search_vector
        ON document_chunks
        USING GIN (search_vector)
    """
    cur.execute(create_search_vector_index_query)
    conn.commit()
    print("Search vector index created or already exists.")
  except Exception as e:
    print(f"Error creating search vector index: {e}")

  finally:
    if conn:
      conn.close()

#add_search_vector()

search_vector column added or already exists.
Search vector index created or already exists.


### Verify Data Insertion

Let's quickly query the database to ensure our data has been successfully stored.

In [41]:
def verify_data_in_db():
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()

        cur.execute("""
        SELECT
            chunk_id,
            text,
            page_numbers,
            source_header,
            vector_dims(embedding),
            search_vector IS NOT NULL -- Check if search_vector is present and not null
        FROM document_chunks
        LIMIT 1;
        """)

        sample_chunk = cur.fetchone()

        if sample_chunk:
            print("Sample chunk:")
            print(f"Chunk ID: {sample_chunk[0]}")
            print(f"Text: {sample_chunk[1][:100]}...")
            print(f"Pages: {sample_chunk[2]}")
            print(f"Header: {sample_chunk[3]}")
            print(f"Embedding dimension: {sample_chunk[4]}")
            print(f"Search Vector exists and is populated: {sample_chunk[5]}") # Display search_vector status
        print(type(sample_chunk[4]))

    except Exception as e:
        print(e)
    finally:
        if conn:
            conn.close()

verify_data_in_db()

Sample chunk:
Chunk ID: 4bb68515-1cd2-4e7e-a146-59bbb510bed3
Text: IN NO EVENT SHALL THE UNIVERSITY OF CALIFORNIA BE LIABLE TO ANY PARTY FOR DIRECT, INDIRECT, SPECIAL,...
Pages: [1, 2]
Header: The PostgreSQL Global Development Group
Embedding dimension: 384
Search Vector exists and is populated: True
<class 'int'>


### Estimate Storage Usage for Embeddings

In [42]:
import sys

# Assuming each float (vector dimension) is 4 bytes (standard for float32/real in PostgreSQL)
# And each embedding has 3072 dimensions for models/gemini-embedding-001
BYTES_PER_FLOAT = 4 # float4 or real type in PostgreSQL

# The number of text chunks with embeddings
num_chunks = len(final_text_chunks_with_embeddings)

# Calculate estimated size for embeddings only
estimated_embedding_bytes = num_chunks * EMBEDDING_DIMENSION * BYTES_PER_FLOAT
estimated_embedding_gb = estimated_embedding_bytes / (1024**3)

print(f"Number of text chunks with embeddings: {num_chunks}")
print(f"Estimated storage for embeddings: {estimated_embedding_gb:.4f} GB")

# Also consider the text content itself and other metadata
# This is a rough estimation; actual storage might vary due to database overhead, indexing, etc.

# Estimate average text length
# We'll calculate the average length of the raw text stored for each chunk
if num_chunks > 0:
    total_text_length = sum(len(chunk.text) for chunk in final_text_chunks_with_embeddings)
    average_text_length = total_text_length / num_chunks
    print(f"Average text length per chunk: {average_text_length:.2f} characters")

    # Assuming 1 byte per character for ASCII or average for UTF-8
    estimated_text_bytes = total_text_length * 1
    estimated_text_gb = estimated_text_bytes / (1024**3)
    print(f"Estimated storage for text content: {estimated_text_gb:.4f} GB")

    total_estimated_gb = estimated_embedding_gb + estimated_text_gb
    print(f"Total estimated data storage (embeddings + text): {total_estimated_gb:.4f} GB")

    if total_estimated_gb > 0.5:
        print("\nBased on this estimate, your data might exceed Neon's 0.5 GB free plan storage limit.")
    else:
        print("\nBased on this estimate, your data should fit within Neon's 0.5 GB free plan storage limit.")
else:
    print("No chunks with embeddings to estimate storage.")

Number of text chunks with embeddings: 15450
Estimated storage for embeddings: 0.0221 GB
Average text length per chunk: 475.24 characters
Estimated storage for text content: 0.0068 GB
Total estimated data storage (embeddings + text): 0.0289 GB

Based on this estimate, your data should fit within Neon's 0.5 GB free plan storage limit.


## Training a Model to Answer Questions

### Using an Open-Source LLM

In [43]:
# Install necessary libraries
!pip install transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time # For measuring latency
import traceback # For debugging

In [44]:
llm_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(llm_model_name, token=HF_TOKEN)

# Ensure you have a GPU runtime enabled for this cell to run efficiently
# Using bfloat16 for reduced memory usage while maintaining reasonable precision
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

## Building our AI Assistant with Retrieval Augmented Generation (RAG)

With text chunks and their embeddings stored in a PostgreSQL vector database, we're ready to build an AI assistant using a Retrieval Augmented Generation (RAG) approach. This typically involves the following key steps:

1.  **Query Embedding**: When a user asks a question, we'll first generate an embedding for their query using the same embedding model (`BAAI/bge-small-en-v1.5`) we used for our document chunks.

2.  **Vector Search (Retrieval)**: We'll then use this query embedding to perform a similarity search in our `document_chunks` table. This will retrieve the top `k` most semantically similar text chunks from our PostgreSQL database (using `pgvector`'s similarity operators).

3.  **Prompt Augmentation**: The retrieved text chunks (which contain the relevant context from our documentation) will be combined with the original user's query. This creates a more informative and context-rich prompt for our Language Model (LLM).

    *Example Prompt Structure:*
    ```
    You are an expert on PostgreSQL documentation. Answer the following question based only on the provided context. If the answer is not in the context, state that you don't know based on the documentation.

    Context:
    [Retrieved Chunk 1 Text]
    [Retrieved Chunk 2 Text]
    ...
    [Retrieved Chunk k Text]

    Question: [User's Original Query]
    ```

4.  **Response Generation (LLM)**: This augmented prompt is then fed to a powerful LLM (like Google's Gemini Pro or another suitable model). The LLM uses the provided context to generate a precise and relevant answer to the user's question.

5.  **Iteration and Refinement**: This is an iterative process. We'll want to:
    *   **Evaluate**: Test our assistant with various queries and assess the quality of the retrieved chunks and the generated answers.
    *   **Refine Chunking Strategy**: Adjust `chunk_size` and `chunk_overlap`, or explore more sophisticated chunking methods if we find context is being missed or split inappropriately.
    *   **Improve Classification**: Further refine our `classify_text_element` and `classify_block_element` functions to better categorize content, which can improve retrieval.
    *   **Prompt Engineering**: Experiment with different prompt structures and instructions to the LLM to guide its response generation.
    *   **Reranking (Optional)**: For advanced RAG, we might introduce a reranking step after initial vector search to further prioritize the most relevant retrieved documents before augmentation.

In [63]:
global_top_k=5

In [64]:
def vector_search(query_embedding, top_k=global_top_k, distance_threshold=0.8, filter_block_types=None):
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()

        # Build the WHERE clause for filtering by block_type
        where_clause = ""
        if filter_block_types:
            # Ensure filter_block_types is a tuple or list for the IN clause
            if isinstance(filter_block_types, str):
                filter_block_types = (filter_block_types,)
            placeholders = ', '.join(['%s'] * len(filter_block_types))
            where_clause = f"WHERE block_type IN ({placeholders})"

        # Perform a vector similarity search
        # The <-> operator computes Euclidean distance. ORDER BY ASC for closest matches.
        # The LIMIT k clause restricts the number of results.
        query_sql = f"""
            SELECT
                chunk_id,
                text,
                page_numbers,
                source_header,
                source_subheader,
                block_type,
                embedding <-> %s AS distance
            FROM
                document_chunks
            {where_clause}
            ORDER BY
                distance ASC
            LIMIT %s;
        """

        # Prepare parameters for the query
        # Explicitly cast the query_embedding to a Vector object

        params = [Vector(query_embedding)]

        if filter_block_types:
            params.extend(filter_block_types)
        params.append(top_k)

        cur.execute(query_sql, params)
        results = cur.fetchall()

        # Convert results to a more readable format (list of dictionaries)
        search_results = []
        for row in results:
          if row[6] <= distance_threshold:
            search_results.append({
                'chunk_id': row[0],
                'text': row[1],
                'page_numbers': row[2],
                'source_header': row[3],
                'source_subheader': row[4],
                'block_type': row[5],
                'distance': row[6]
            })
        return search_results

    except Exception as e:
        print("Error during vector_search")
        traceback.print_exc()
        return None
    finally:
        if conn:
            conn.close()

In [65]:
def keyword_search(keyword_query, top_k=global_top_k):
  conn = None
  try:
    conn = get_db_connection()
    cur = conn.cursor()
    # Perform keyword search
    cur.execute("""
      SELECT
          chunk_id,
          text,
          page_numbers,
          source_header,
          source_subheader,
          block_type,
          ts_rank(search_vector, plainto_tsquery('english', %s)) AS rank
      FROM document_chunks
      WHERE search_vector @@ plainto_tsquery('english', %s)
      ORDER BY rank DESC
      LIMIT %s;
      """, (keyword_query, keyword_query, top_k))
    results = cur.fetchall()

    keyword_search_results = []
    for row in results:
        keyword_search_results.append({
            'chunk_id': row[0],
            'text': row[1],
            'page_numbers': row[2],
            'source_header': row[3],
            'source_subheader': row[4],
            'block_type': row[5],
            'rank': row[6] # Include the rank
        })
    return keyword_search_results

  except Exception as e:
    print("Error during keyword_search")
    traceback.print_exc()
    return None
  finally:
    if conn:
      conn.close()

In [88]:
def get_relevant_pages(relevant_chunks):
  # Collect all page numbers, maintaining relevance order and ensuring uniqueness.
  ordered_unique_pages = []
  seen_pages = set()

  for chunk in relevant_chunks:
    if chunk.get('page_numbers'):
      for page_num in chunk['page_numbers']:
        if page_num not in seen_pages:
          ordered_unique_pages.append(page_num)
          seen_pages.add(page_num)

  return ordered_unique_pages

In [98]:
def setup_prompt(query_text, relevant_chunks):
  try:
    prompt = f"""
    You are an expert on PostgreSQL documentation.

    Answer the question based *solely* on the provided 'Retrieved Documentation' if possible.
    If the 'Retrieved Documentation' does not contain enough information to fully answer the question,
    then use your general knowledge of PostgreSQL.

    Always clearly state the source of your answer at the end, choosing *one* of the following two options:
    1. 'Answer Source: PostgreSQL Documentation.' (If the answer is derived primarily or entirely from the provided documentation.)
    2. 'Answer Source: General PostgreSQL Knowledge.' (If the answer relies significantly on knowledge outside the provided documentation.)

    Never invent documentation that was not retrieved.
    When answering, synthesize information from *all* relevant retrieved chunks to provide a comprehensive answer, especially if the query has multiple parts.

    """
    if relevant_chunks:
      prompt += "\nRetrieved Documentation:\n\n"
      for i, chunk in enumerate(relevant_chunks, 1):
          prompt += (
              f"[Chunk {i}]\n"
              f"Header: {chunk['source_header']}\n"
              f"Text:\n{chunk['text']}\n\n"
          )
    prompt += f"\nQuestion: {query_text}\n"
    return prompt
  except Exception as e:
    print("Error in setup_prompt")
    traceback.print_exc()
    return None

In [112]:
import re # Import regex for stripping introductory phrases

def get_answer(query_text, llm_model, top_k=global_top_k, use_hybrid_search=True, debug=False):
  # Call the LLM to answer the question from setup_prompt
  start_time = time.time()
  try:
    #print("Generating embedding...")
    query_embedding = get_embedding(query_text)
    query_embedding_end_time = time.time()
    query_embedding_latency = query_embedding_end_time - start_time

    relevant_chunks = []
    if use_hybrid_search:
        # Perform vector search
        vector_results = []
        if query_embedding is not None:
            vector_results = vector_search(query_embedding, top_k=top_k)
        if not vector_results:
            vector_results = []

        # Perform keyword search
        keyword_results = keyword_search(query_text, top_k=top_k)
        if not keyword_results:
            keyword_results = []

        # Combine and de-duplicate results
        combined_chunks_map = {chunk['chunk_id']: chunk for chunk in vector_results}
        for chunk in keyword_results:
            # If a chunk_id already exists from vector search, we keep the vector search result
            # to prioritize more complete metadata (distance is present).
            if chunk['chunk_id'] not in combined_chunks_map:
                combined_chunks_map[chunk['chunk_id']] = chunk

        # Convert back to list and sort.
        # Prioritize chunks with a 'distance' (from vector search), then by 'rank' (from keyword search).
        def sort_combined_chunks(chunk_item):
            # Prioritize chunks that came from vector search (have 'distance')
            has_distance = 'distance' in chunk_item and chunk_item['distance'] is not None
            distance_val = chunk_item.get('distance', float('inf')) # Lower distance is better

            # Prioritize chunks that came from keyword search (have 'rank')
            has_rank = 'rank' in chunk_item and chunk_item['rank'] is not None
            rank_val = chunk_item.get('rank', 0.0) # Higher rank is better

            # Sorting heuristic:
            # 1. Chunks with valid distance first (smaller distance is better)
            # 2. For chunks without distance, prioritize by rank (larger rank is better)
            # This makes vector search dominant, then keyword search.
            if has_distance:
                return (0, distance_val, -rank_val) # (flag for has_distance, distance_asc, rank_desc)
            elif has_rank:
                return (1, float('inf'), -rank_val) # (flag for no_distance, large_distance, rank_desc)
            else:
                return (2, float('inf'), 0.0) # (flag for no_distance_no_rank, large_distance, zero_rank)

        relevant_chunks = sorted(list(combined_chunks_map.values()), key=sort_combined_chunks)[:top_k]

    elif query_embedding is not None:
        # Fallback to pure vector search if hybrid search is off or keyword query is empty
        relevant_chunks = vector_search(query_embedding, top_k=top_k)
    elif query_text is not None: # Fallback to pure keyword search if no embedding or no hybrid search
        relevant_chunks = keyword_search(query_text, top_k=top_k)

    retrieval_end_time = time.time()
    retrieval_latency = retrieval_end_time - query_embedding_end_time

    if not relevant_chunks:
      print("No relevant chunks found.")

    # Get the complete prompt that will be sent to the LLM
    full_prompt_sent_to_llm = setup_prompt(query_text, relevant_chunks)

    if debug:
      # --- DEBUGGING: Print relevant_chunks and full_prompt_sent_to_llm ---
      print("\n--- Retrieved Chunks (for debugging) ---")
      for i, chunk in enumerate(relevant_chunks):
          print(f"Chunk {i+1}:")
          print(f"  Chunk ID: {chunk.get('chunk_id')}")
          print(f"  Block Type: {chunk.get('block_type')}")
          print(f"  Header: {chunk.get('source_header')}")
          print(f"  Subheader: {chunk.get('source_subheader')}")
          print(f"  Pages: {chunk.get('page_numbers')}")
          print(f"  Text (first 150 chars): {chunk.get('text', '')[:150]}...")
          if 'distance' in chunk: print(f"  Distance: {chunk['distance']}")
          if 'rank' in chunk: print(f"  Rank: {chunk['rank']}")
          print("--------------------------------------------------")
      print("\n--- Full Prompt Sent to LLM (for debugging) ---")
      print(full_prompt_sent_to_llm)
      print("\n--- END DEBUGGING ---")
      # --- END DEBUGGING

    # Pass prompt to an LLM
    #print("Passing prompt to LLM")
    input_ids = tokenizer(full_prompt_sent_to_llm, return_tensors="pt").to(llm_model.device)

    # Generate outputs. The generated_ids will contain the input_ids followed by the new tokens.
    #print("Generating output")
    generated_ids = llm_model.generate(**input_ids, max_new_tokens=500, pad_token_id=tokenizer.eos_token_id)

    # Decode only the newly generated part of the response (after the input_ids)
    #print("Decoding response")
    prompt_length = input_ids["input_ids"].shape[1]

    new_tokens = generated_ids[:, prompt_length:]

    llm_answer = tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0].strip()

    #print("Getting relevant page numbers")
    relevant_page_numbers = get_relevant_pages(relevant_chunks)
    if not relevant_page_numbers:
      llm_answer += "\nNo relevant pages found."
    else:
      llm_answer += f'\nRelevant page numbers of PostgreSQL 18 Official Documentation: {relevant_page_numbers}\n'
    llm_answer_end_time = time.time()
    llm_answer_latency = llm_answer_end_time - retrieval_end_time
    print(f"Query embedding latency: {query_embedding_latency:.3f} seconds")
    print(f"Retrieval latency: {retrieval_latency:.3f} seconds")
    print(f"LLM latency: {llm_answer_latency:.3f} seconds")
    print(f"Total latency: {llm_answer_end_time - start_time:.3f} seconds\n")
    return llm_answer
  except Exception as e:
    print("Error in get_answer")
    traceback.print_exc()
    return None

Now, let's test the `get_answer` function with an example query, using our newly integrated Hugging Face model.

## Test queries with RAG model

In [124]:
def test_answer_generation(query_text, llm_model):
  try:
    answer = get_answer(query_text, llm_model, debug=True)
    print(answer)
  except Exception as e:
    print("Error in test_answer_generation")
    traceback.print_exc()

In [125]:
query_1 = "How to create a database in PostgreSQL?"
test_answer_generation(query_1, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 0f69ea1b-b483-4ad1-9b0c-db6d9db6b5b8
  Block Type: body_block
  Header: createdb
  Subheader: None
  Pages: [2288]
  Text (first 150 chars): createdb — create a new PostgreSQL database...
  Distance: 0.42553196180056757
--------------------------------------------------
Chunk 2:
  Chunk ID: 5313f523-4a2c-48f8-a7ef-f38d3c8fa64a
  Block Type: query_block
  Header: Description
  Subheader: None
  Pages: [1907]
  Text (first 150 chars): CREATE DATABASE creates a new PostgreSQL database.

To create a database, you must be a superuser or have the special CREATEDB privilege. See CREATE
R...
  Distance: 0.4733383679234136
--------------------------------------------------
Chunk 3:
  Chunk ID: 97c90874-471f-4a29-8fd7-65c0daa5122a
  Block Type: body_block
  Header: 1.3. Creating a Database
  Subheader: None
  Pages: [44, 45]
  Text (first 150 chars): You can also create databases with other names. PostgreSQL allows you to create any 

In [126]:
query_2 = "What is the difference between VACUUM and VACUUM FULL?"
answer = get_answer(query_2, llm_model)
print(answer)

Query embedding latency: 0.013 seconds
Retrieval latency: 4.042 seconds
LLM latency: 18.880 seconds
Total latency: 22.936 seconds

Answer Source: PostgreSQL Documentation.

Answer:
VACUUM and VACUUM FULL are two variants of the vacuum command in PostgreSQL. The main difference between them lies in their functionality and performance.

VACUUM is the standard form of vacuuming, which reclaims space by removing dead row versions in tables and indexes and marks the space available for future reuse. It does not return the space to the operating system, but keeps it available for reuse within the same table. VACUUM can operate in parallel with normal reading and writing of the table, as it does not require an exclusive lock. However, it does not actively compact tables or return unused space to the operating system.

VACUUM FULL, on the other hand, actively compacts tables by writing a complete new version of the table file with no dead space. This minimizes the size of the table and returns

In [127]:
query_3 = """How can I improve the performance of the following SQL query?
WITH start_date AS
  SELECT MIN(DATE_TRUNC('DAY', days))
SELECT day, DATE_DIFF('DAY', day, start_date) AS days_from_start
FROM date_table
"""

answer = get_answer(query_3, llm_model)
print(answer)

Query embedding latency: 0.014 seconds
Retrieval latency: 4.122 seconds
LLM latency: 23.907 seconds
Total latency: 28.042 seconds

WHERE DATE_TRUNC('MONTH', day) = '2020-01-01'::timestamp
GROUP BY day
ORDER BY days_from_start;

Answer Source: General PostgreSQL Knowledge.

The provided query calculates the number of days between a start date and each day in the 'date_table' where the month matches '2020-01-01'. It uses a common table expression (CTE) to first find the minimum date for the month of January 2020.

To improve the performance of this query, you can create statistics on the 'date_table' for the month and day expressions. This will help the PostgreSQL query optimizer to make better execution plans.

First, analyze the table to gather statistics:

```sql
ANALYZE date_table;
```

Then, create statistics on the 'date_trunc' expressions:

```sql
CREATE STATISTICS date_stat (ndistinct) ON date_trunc('month', date_column)
FROM date_table;
CREATE STATISTICS day_stat (ndistinct) ON 

In [128]:
query_4 = "Explain the purpose of the PostgreSQL WAL (Write-Ahead Log)."
test_answer_generation(query_4, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 1820e25e-a45b-43b5-a59c-c059e5f07c76
  Block Type: body_block
  Header: Extensions
  Subheader: None
  Pages: [2772]
  Text (first 150 chars): Certain extensions, principally extensions that implement custom access methods, may need to perform
write-ahead  logging  in  order  to  ensure  cras...
  Distance: 0.5128378824278748
--------------------------------------------------
Chunk 2:
  Chunk ID: 6f3159c2-3630-4a55-a9f9-16894c4c4117
  Block Type: body_block
  Header: O.3. pg_xlogdump renamed to pg_waldump
  Subheader: None
  Pages: [3271]
  Text (first 150 chars): PostgreSQL 9.6 and below provided a command named pg_xlogdump  to read write-ahead-log (WAL)
files. This command was renamed to pg_waldump, see pg_wal...
  Distance: 0.5347656439480942
--------------------------------------------------
Chunk 3:
  Chunk ID: a4733e10-f93e-4256-b658-04e0829ec098
  Block Type: body_block
  Header: Description
  Subheader: None
  Pages

In [129]:
query_5 = "What are some common causes of performance issues in PostgreSQL and how can they be mitigated?"
test_answer_generation(query_5, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 48b05126-6a11-4b5f-a6ef-f5905c531d2c
  Block Type: body_block
  Header: 18.4.4. Linux Memory Overcommit
  Subheader: None
  Pages: [690, 691]
  Text (first 150 chars): If PostgreSQL itself is the cause of the system running out of memory, you can avoid the problem by
changing your configuration. In some cases, it may...
  Distance: 0.6266782163847522
--------------------------------------------------
Chunk 2:
  Chunk ID: a91fef9f-e58f-4c64-830d-0feab83fbdbd
  Block Type: body_block
  Header: Chapter 14. Performance Tips
  Subheader: None
  Pages: [598]
  Text (first 150 chars): Query performance can be affected by many things. Some of these can be controlled by the user, while
others are fundamental to the underlying design o...
  Distance: 0.6332812045210514
--------------------------------------------------
Chunk 3:
  Chunk ID: 6ba4b6e4-d277-4a95-a4d6-ca675793c1ad
  Block Type: body_block
  Header: E.4.2. Changes
  Subhea

In [130]:
query_6 = "How do I back up and restore a PostgreSQL database?"
test_answer_generation(query_6, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 084603c0-4dee-41c5-b4f1-e79fd5772f67
  Block Type: body_block
  Header: pg_restore
  Subheader: None
  Pages: [2393]
  Text (first 150 chars): pg_restore — restore a PostgreSQL database from an archive file created by pg_dump...
  Distance: 0.5696147935918533
--------------------------------------------------
Chunk 2:
  Chunk ID: 3ced3562-8875-49ad-87a5-61fa24c733f7
  Block Type: body_block
  Header: 18.6.1. Upgrading Data via pg_dumpall
  Subheader: None
  Pages: [694, 695]
  Text (first 150 chars): To back up your database installation, type:

pg_dumpall > outputfile

To make the backup, you can use the pg_dumpall command from the version you are...
  Distance: 0.5824049191785925
--------------------------------------------------
Chunk 3:
  Chunk ID: a16ee5c0-9843-4940-84ae-de868ce3c474
  Block Type: body_block
  Header: pg_basebackup
  Subheader: None
  Pages: [2312]
  Text (first 150 chars): pg_basebackup — take a base 

In [131]:
query_7 = "Describe the different types of indexes available in PostgreSQL and when to use each."
test_answer_generation(query_7, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 0e8717fe-93c9-4df0-bf4f-98a04615aff2
  Block Type: body_block
  Header: 11.2. Index Types
  Subheader: None
  Pages: [522]
  Text (first 150 chars): PostgreSQL provides several index types: B-tree, Hash, GiST, SP-GiST, GIN, BRIN, and the extension
bloom. Each index type uses a different algorithm t...
  Distance: 0.5197032557683701
--------------------------------------------------
Chunk 2:
  Chunk ID: ab36deac-28a4-4b43-aac0-4caf14e5ed8e
  Block Type: body_block
  Header: Interface Definition
  Subheader: None
  Pages: [2755]
  Text (first 150 chars): This chapter defines the interface between the core PostgreSQL system and index access methods, which
manage individual index types. The core system k...
  Distance: 0.5332307633723011
--------------------------------------------------
Chunk 3:
  Chunk ID: a966a1ac-8010-4012-8efa-040f7f7c5e7a
  Block Type: body_block
  Header: 11.1. Introduction
  Subheader: None
  Pages: [52

In [132]:
query_8 = "What is connection pooling in PostgreSQL and why is it beneficial?"
test_answer_generation(query_8, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: bf24952c-cb9b-48af-a8a2-81adbbb00606
  Block Type: body_block
  Header: 1.2. Architectural Fundamentals
  Subheader: None
  Pages: [43]
  Text (first 150 chars): The PostgreSQL server can handle multiple concurrent connections from clients. To achieve this it starts
(“forks”) a new process for each connection. ...
  Distance: 0.6445901092472344
--------------------------------------------------
Chunk 2:
  Chunk ID: 62301e1a-c039-48d0-9e47-16687ce36076
  Block Type: body_block
  Header: 19.3.1. Connection Settings
  Subheader: None
  Pages: [710, 711, 712]
  Text (first 150 chars): max_connections (integer)

Determines the maximum number of concurrent connections to the database server. The default is
typically 100 connections, b...
  Distance: 0.6684820645579094
--------------------------------------------------
Chunk 3:
  Chunk ID: e66090ef-1a87-4754-9b2e-d98086982188
  Block Type: body_block
  Header: Chapter 47. Logical 

In [133]:
query_9 = """
Given a table Employee with columns:
employeeID INT
name VARCHAR
managerid INT
Write query to find all employees who do not have a manager
"""
test_answer_generation(query_9, llm_model)



--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: c10c2e8b-4112-45fd-b954-2455b3674d1f
  Block Type: body_block
  Header: Examples
  Subheader: None
  Pages: [2241, 2242, 2243, 2244, 2245]
  Text (first 150 chars): This example uses WITH RECURSIVE to find all subordinates (direct or indirect) of the employee Mary,
and their level of indirectness, from a table tha...
  Distance: 0.7157700587348941
--------------------------------------------------
Chunk 2:
  Chunk ID: 7d7bef6e-f671-4fae-aa1a-c5770cb0815e
  Block Type: body_block
  Header: Synopsis
  Subheader: None
  Pages: [2248]
  Text (first 150 chars): [ WITH [ RECURSIVE ] with_query [, ...] ]
SELECT [ ALL | DISTINCT [ ON ( expression [, ...] ) ] ]
[ { * | expression [ [ AS ] output_name ] } [, ...] ...
  Distance: 0.7168348754455871
--------------------------------------------------
Chunk 3:
  Chunk ID: 149dcd64-9c70-4378-bc5e-c051a89c14cc
  Block Type: body_block
  Header: Synopsis
  Subheader: None
  Pages: [2225, 22

In [134]:
query_10 = """
Given a table Product with columns:
productID INT
storeID INT
name VARCHAR
price DECIMAL
and a table Store with columns:
storeID INT
name VARCHAR
Write a query to find the store name and price of each product at the store where it is most expensive
"""

test_answer_generation(query_10, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 2acf3044-56ef-4184-a454-cb8ffa70e302
  Block Type: body_block
  Header: 5.1. Table Basics
  Subheader: None
  Pages: [104, 105]
  Text (first 150 chars): Of course, the previous example was heavily contrived. Normally, you would give names to your tables
and columns that convey what kind of data they st...
  Distance: 0.6622525400766094
--------------------------------------------------
Chunk 2:
  Chunk ID: 22b19890-b006-41de-86ea-0da5d06cedff
  Block Type: body_block
  Header: 5.5.1. Check Constraints
  Subheader: None
  Pages: [109, 110, 111, 112]
  Text (first 150 chars): CREATE TABLE products (
product_no integer,
name text,
price numeric,
CHECK (price > 0),
discounted_price numeric,
CHECK (discounted_price > 0),
CHECK...
  Distance: 0.6717654294114909
--------------------------------------------------
Chunk 3:
  Chunk ID: 0a5512c5-3009-4425-ab61-96d26684e7ae
  Block Type: note_block
  Header: 6.1. Inserting Data
  Subh

In [135]:
query_11 = "Explain how to use LATERAL JOIN in PostgreSQL and provide an example."
test_answer_generation(query_11, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: ad30927b-8520-4860-b568-c430cb28f88d
  Block Type: body_block
  Header: 7.2.1. The FROM Clause
  Subheader: 7.2.1.5. LATERAL Subqueries
  Pages: [167, 175, 176]
  Text (first 150 chars): or in several other equivalent formulations. (As already mentioned, the LATERAL key word is unnecessary
in this example, but we use it for clarity.)
I...
  Distance: 0.546206283434928
--------------------------------------------------
Chunk 2:
  Chunk ID: a041deb0-921a-4f23-9140-896706bf23ee
  Block Type: query_block
  Header: 7.2.1. The FROM Clause
  Subheader: 7.2.1.5. LATERAL Subqueries
  Pages: [167, 175, 176]
  Text (first 150 chars): When a FROM item contains LATERAL cross-references, evaluation proceeds as follows: for each row of
the FROM item providing the cross-referenced colum...
  Distance: 0.6234605426402039
--------------------------------------------------
Chunk 3:
  Chunk ID: 687a1e9b-be77-49e9-90f3-327c9922dcc3
  Block Type

In [136]:
query_12 = "How do EXISTS and IN subqueries differ in performance and usage?"
test_answer_generation(query_12, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: cb8557b2-d674-47ae-8eae-c4c0251fb023
  Block Type: body_block
  Header: 9.24.1. EXISTS
  Subheader: None
  Pages: [450, 451]
  Text (first 150 chars): EXISTS (subquery)

The argument of EXISTS is an arbitrary SELECT statement, or subquery. The subquery is evaluated to
determine whether it returns any...
  Distance: 0.5307712641807825
--------------------------------------------------
Chunk 2:
  Chunk ID: 5399064f-bf39-4225-854c-8d1a9e837238
  Block Type: body_block
  Header: 9.24.1. EXISTS
  Subheader: None
  Pages: [450, 451]
  Text (first 150 chars): Since the result depends only on whether any rows are returned, and not on the contents of those rows, the
output list of the subquery is normally uni...
  Distance: 0.6038073726507824
--------------------------------------------------
Chunk 3:
  Chunk ID: 193d5e09-b2f0-4eb8-aeac-664fddfa18c3
  Block Type: body_block
  Header: 9.24.2. IN
  Subheader: None
  Pages: [451]
  Tex

In [143]:
query_13 = "What are window functions in PostgreSQL and when should I use ROW_NUMBER() versus RANK()?"
test_answer_generation(query_13, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: b2831a4d-bb53-4f2a-bbf8-df309ae4f553
  Block Type: body_block
  Header: 9.22. Window Functions
  Subheader: Table 9.67. General-Purpose Window Functions
  Pages: [448, 449]
  Text (first 150 chars): Function
Description

row_number () → bigint
Returns the number of the current row within its partition, counting from 1.

rank () → bigint
Returns th...
  Distance: 0.623141434050077
--------------------------------------------------
Chunk 2:
  Chunk ID: 521c4c43-7aa9-49ca-813c-d01d5ec7f59b
  Block Type: body_block
  Header: F.27.2. Sample Output
  Subheader: None
  Pages: [3132, 3133]
  Text (first 150 chars): postgres=# SELECT * FROM pg_freespace('foo');
blkno | avail
-------+-------
0 |     0
1 |     0
2 |     0
3 |    32
4 |   704
5 |   704
6 |   704
7 | ...
  Distance: 0.6257118224683689
--------------------------------------------------
Chunk 3:
  Chunk ID: 889d340a-142a-4bf4-a0ad-f845c824fb19
  Block Type: body_block
  H

In [144]:
query_14 = "What are window functions in PostgreSQL and when should I use ROW_NUMBER() versus RANK()? Furthermore, when should I use functions like LEAD() and LAG()?"
test_answer_generation(query_14, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 511f0b30-94f6-446f-a315-09332f7a5322
  Block Type: body_block
  Header: Chapter 9. Functions and Operators
  Subheader: None
  Pages: [276]
  Text (first 150 chars): PostgreSQL provides a large number of functions and operators for the built-in data types. This chapter
describes most of them, although additional sp...
  Distance: 0.6314716497628323
--------------------------------------------------
Chunk 2:
  Chunk ID: 6af6766b-7167-43df-8eff-c41acdf945f4
  Block Type: body_block
  Header: 4.2.8. Window Function Calls
  Subheader: None
  Pages: [92, 93, 94]
  Text (first 150 chars): The built-in window functions are described in Table 9.67. Other window functions can be added by the
user. Also, any built-in or user-defined general...
  Distance: 0.6329823489718768
--------------------------------------------------
Chunk 3:
  Chunk ID: 9a23f2d7-718a-4ac4-a5b2-4aeba2fe3068
  Block Type: body_block
  Header: 9.26. Set Returnin

In [145]:
query_15 = "How can I query JSONB data in PostgreSQL to find documents where a specific key has a certain value?"
test_answer_generation(query_15, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 1b0dee6c-a0ea-488a-b0f0-f7fd17d69fed
  Block Type: output_block
  Header: 9.16.2. The SQL/JSON Path Language
  Subheader: None
  Pages: [404, 405, 406, 407]
  Text (first 150 chars): SELECT '{
"track": {
"segments": [
{
"location":   [ 47.763, 13.4034 ],
"start time": "2018-10-14 10:05:14",
"HR": 73
},
{
"location":   [ 47.706, 13....
  Distance: 0.6437082758594476
--------------------------------------------------
Chunk 2:
  Chunk ID: e473f258-451a-481e-bc27-c7164df89135
  Block Type: body_block
  Header: 9.16.2. The SQL/JSON Path Language
  Subheader: None
  Pages: [404, 405, 406, 407]
  Text (first 150 chars): => select jsonb_path_query(:'json', '$.track.segments[*].location');
jsonb_path_query
-------------------
[47.763, 13.4034]
[47.706, 13.2635]...
  Distance: 0.6452373913519973
--------------------------------------------------
Chunk 3:
  Chunk ID: f4b0a694-898e-4fda-935d-73c61604e714
  Block Type: query_block
  Hea

In [146]:
query_16 = "Describe PostgreSQL's transaction isolation levels and their implications."
test_answer_generation(query_16, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: f2dc7903-c6b1-45ad-a7f7-48bab7f669e6
  Block Type: body_block
  Header: 13.2. Transaction Isolation
  Subheader: None
  Pages: [582, 583]
  Text (first 150 chars): The SQL standard and PostgreSQL-implemented transaction isolation levels are described in Table 13.1....
  Distance: 0.41181126881489344
--------------------------------------------------
Chunk 2:
  Chunk ID: 6a3c1613-5800-4d5a-9980-01818508a708
  Block Type: body_block
  Header: 13.2. Transaction Isolation
  Subheader: Table 13.1. Transaction Isolation Levels
  Pages: [582, 583]
  Text (first 150 chars): Serializable
Not possible
Not possible
Not possible
Not possible

In PostgreSQL, you can request any of the four standard transaction isolation levels...
  Distance: 0.5648807898653313
--------------------------------------------------
Chunk 3:
  Chunk ID: 956842a2-abab-46ec-9fb1-1a77090d78a2
  Block Type: body_block
  Header: Description
  Subheader: None
  Pag

In [147]:
query_17 = "How do I grant read-only access to a specific schema for a new user role?"
test_answer_generation(query_17, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 8eaac3ca-909c-4b2b-91b6-75246cdb1aea
  Block Type: body_block
  Header: Examples
  Subheader: None
  Pages: [2083, 2084]
  Text (first 150 chars): Grant all privileges on all views in schema public to role webuser:

DO $$DECLARE r record;
BEGIN
FOR r IN SELECT table_schema, table_name FROM
inform...
  Distance: 0.618700659354811
--------------------------------------------------
Chunk 2:
  Chunk ID: 0518e56d-1c93-4995-9805-4ae057db683a
  Block Type: body_block
  Header: Synopsis
  Subheader: None
  Pages: [1741, 1742]
  Text (first 150 chars): ALTER DEFAULT PRIVILEGES
[ FOR { ROLE | USER } target_role [, ...] ]
[ IN SCHEMA schema_name [, ...] ]
abbreviated_grant_or_revoke

where abbreviated_...
  Distance: 0.6267090794823652
--------------------------------------------------
Chunk 3:
  Chunk ID: 041b6790-477c-4dd6-9cee-3b728d827983
  Block Type: body_block
  Header: Synopsis
  Subheader: None
  Pages: [2158, 2159]
  Text (f

In [148]:
query_18 = "What is the pg_stat_statements extension and how can I use it to analyze query performance?"
test_answer_generation(query_18, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 7bf49d03-b904-405d-a51b-c742f8c07c4a
  Block Type: body_block
  Header: SQL planning and execution
  Subheader: None
  Pages: [3142]
  Text (first 150 chars): The pg_stat_statements module provides a means for tracking planning and execution statistics of
all SQL statements executed by a server.

The module ...
  Distance: 0.5266026976775263
--------------------------------------------------
Chunk 2:
  Chunk ID: 5422a809-37e5-4a14-9718-bbfb92d52247
  Block Type: body_block
  Header: F.32.1. The pg_stat_statements View
  Subheader: None
  Pages: [3142]
  Text (first 150 chars): The statistics gathered by the module are made available via a view named pg_stat_statements.
This view contains one row for each distinct combination...
  Distance: 0.5585003487988017
--------------------------------------------------
Chunk 3:
  Chunk ID: 393512df-35fa-4cfc-b7ce-c1defb4353ea
  Block Type: note_block
  Header: F.32.1. The pg_stat_stat

In [149]:
query_19 = "How do I create a BEFORE INSERT trigger in PostgreSQL and what variables are available within the trigger function?"
test_answer_generation(query_19, llm_model)


--- Retrieved Chunks (for debugging) ---
Chunk 1:
  Chunk ID: 7553a2cb-4736-4e85-9fb4-6a57e449308e
  Block Type: body_block
  Header: 5.12.3. Partitioning Using Inheritance
  Subheader: 5.12.3.1. Example
  Pages: [150, 151, 152, 153]
  Text (first 150 chars): CREATE OR REPLACE FUNCTION measurement_insert_trigger()
RETURNS TRIGGER AS $$
BEGIN
INSERT INTO measurement_y2008m01 VALUES (NEW.*);
RETURN NULL;
END;...
  Distance: 0.5819481399902332
--------------------------------------------------
Chunk 2:
  Chunk ID: 3a094041-e4a2-43d9-88dd-ce834b12b038
  Block Type: body_block
  Header: 42.6. Trigger Functions in PL/Tcl
  Subheader: None
  Pages: [1578, 1579, 1580]
  Text (first 150 chars): Trigger functions can be written in PL/Tcl. PostgreSQL requires that a function that is to be called as a
trigger must be declared as a function with ...
  Distance: 0.594580195413537
--------------------------------------------------
Chunk 3:
  Chunk ID: 42cc1c9f-fd75-4a36-9a80-63ed5d2ce96b
  Block Typ

# Summary

## Deploying the RAG Assistant with Gradio

In [150]:
# Install Gradio library
!pip install gradio

In [151]:
import gradio as gr

def answer_question_gradio(question):
    """
    Wrapper function for Gradio interface to call the RAG model.
    """
    print(f"Received question: {question}")
    try:
        response = get_answer(question, llm_model, debug=False) # Set debug=False for deployment
        if response: # Ensure response is not None
            return response
        else:
            return "I could not generate an answer for that question."
    except Exception as e:
        print(f"Error in answer_question_gradio: {e}")
        return "An error occurred while processing your request."

# Create the Gradio interface
iface = gr.Interface(
    fn=answer_question_gradio,
    inputs=gr.Textbox(lines=2, placeholder="Enter your question about PostgreSQL..."),
    outputs="text",
    title="PostgreSQL AI Documentation Assistant",
    description="Ask any question about PostgreSQL documentation and get an answer backed by RAG."
)

# Launch the Gradio interface
# Set share=True to get a public link (useful for Colab notebooks)
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3ee321732f72bec2ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gemma-2-2b-it does not attempt to reasonably answer query 8 even when the prompt says to use the general knowledge if the provided context can't give a reasonable answer. Gemma-2 also sometimes treats query 3's start_date as a separate column rather than a nested calculation

I then tried Gemma-7b-it but it took way too long to answer simple queries, such as 45 seconds to answer the relatively simple query 1

I chose global_top_k=5 because, when I tried higher values, the last retrieved chunks were often marginally or not relevant to the question being asked.

I note that my model struggles to retrieve answers from documentation when queries are involved, including when analyzing queries or especially when writing queries. This is likely due to queries highly varying in things like column names and sets of functions used and the fact that the documentation doesn't have so many query examples

I considered using objective metrics like recall\@k or precision@\k, but I realized that this would require extensive labor in manually sifting through and identifying documentation / chunks for each query, which I feel is too much for this project due to having over 15,000 chunks and the documentation being over 3300 pages. I also considered using another, more well-known LLM like ChatGPT or Gemini to evaluate my current model, but I noted that these outside LLMs could hallucinate or be biased by the context in my prompts and don't have a ground truth to refer to.

Overall, based on the relevant chunks and answers I am seeing, I am satisfied with this model. I realize that there is a possibility that some optimal chunks are not always being retrieved, but I note that my retrieved chunks are very often very relevant to the queries and that the answers are often based in the retrieved chunks.

To improve this assistant further, I could focus on utilizing outside documentation containing example queries pertaining to specific use cases to help train this assistant.